# Bimodal Gaussian latent generation

Minimal CUDA-only notebook for the 2D bimodal Gaussian toy.
Configs, sampling, the hardcoded encoder/decoder, and plotting helpers live in `src.toy`; the notebook keeps the loss assembly, training loop, and visualizations inline.

In [1]:
from __future__ import annotations

import hashlib
from datetime import datetime
from pathlib import Path

import torch
import torch.nn.functional as F
from IPython.display import display
from jaxtyping import Float
from torch import Tensor
from tqdm.autonotebook import tqdm

from src.toy import (
    BimodalRoundtripModel,
    DataConfig,
    ExperimentConfig,
    LossConfig,
    PlotConfig,
    TrainConfig,
    build_gaussian_contour_points,
    build_latent_score_snapshots,
    estimate_denoising_loss_curve,
    mode_centers_tensor,
    normalize_vectors_for_display,
    plot_bimodal_snapshot,
    plot_denoising_loss_curve,
    plot_latent_score_snapshot_selector,
    plot_pullback_snapshot,
    plot_training_history,
    prior_matching_loss,
    sample_bimodal_gaussian,
    sample_time,
    save_plotly_figure,
)

DEVICE = "cuda:0"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_HASH = hashlib.sha1(
    datetime.now().isoformat(timespec="microseconds").encode()
).hexdigest()[:12]
RUN_CATALOG = f"{RUN_TIMESTAMP}-{RUN_HASH}"
RUN_ROOT = Path("/tmp") / RUN_CATALOG
RUN_ROOT.mkdir(parents=True, exist_ok=True)

CONFIG = ExperimentConfig(
    data=DataConfig(
        batch_size=2048,
        mode_center=(5, 0),
        std_cap=0.35,
        offset=(0.3, 0.4),
    ),
    loss=LossConfig(
        t_min=0.05,
        reconstruction_weight=1.0,
        prior_matching_weight=0.5,
        cycle_data_weight=1.0,
        cycle_prior_weight=1.0,
        denoising_weight=1.0,
        score_weight=0.2,
    ),
    train=TrainConfig(
        steps=1280000,
        lr=5e-4,
        weight_decay=1e-2,
        grad_clip=1.0,
        decoder_attenuation=10.0,
        log_every_steps=100,
    ),
    plot=PlotConfig(
        eval_size=4096,
        latent_plot_size=512,
        contour_levels=(0.1, 0.3, 0.5, 1.0, 2.5, 2.0),
        contour_points_per_level=32,
        snapshot_every_steps=4000,
        arrow_stride=12,
    ),
)

LATENT_DRIFT_ARROW_DISPLAY_LENGTH = 0.32
PULLBACK_ARROW_DISPLAY_LENGTH = 0.44
SCORE_NOISE_LEVELS = (0.05, 0.1, 0.2, 0.3, 0.5, 0.7)
SCORE_ARROW_DISPLAY_LENGTH = 0.32
DENOISING_EVAL_LEVELS = (0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0)
DENOISING_EVAL_NOISE_DRAWS = 16
NOISE_WEIGHTING_LABEL = "uniform"

print(f"device: {DEVICE}")
print(f"run catalog: {RUN_CATALOG}")
print(f"artifacts: {RUN_ROOT}")
CONFIG

/tmp/ipykernel_2517860/3438493345.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


device: cuda:0
run catalog: 20260321-125751-0de54aed5d04
artifacts: /tmp/20260321-125751-0de54aed5d04


ExperimentConfig(data=DataConfig(batch_size=2048, mode_center=(5.0, 0.0), std_cap=0.35, offset=(0.3, 0.4)), loss=LossConfig(t_min=0.05, reconstruction_weight=1.0, prior_matching_weight=0.5, cycle_data_weight=1.0, cycle_prior_weight=1.0, denoising_weight=1.0, score_weight=0.2), train=TrainConfig(steps=1280000, lr=0.0005, weight_decay=0.01, grad_clip=1.0, decoder_attenuation=10.0, log_every_steps=100), plot=PlotConfig(eval_size=4096, latent_plot_size=512, contour_levels=(0.1, 0.3, 0.5, 1.0, 2.5, 2.0), contour_points_per_level=32, snapshot_every_steps=4000, arrow_stride=12))

In [2]:
def analytic_gaussian_denoiser(
    *,
    y_t: Float[Tensor, "batch 2"],
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 2"]:
    denoising_coef = (1.0 - t) / ((1.0 - t).square() + t.square())
    return denoising_coef * y_t


def latent_noise_weight(
    *,
    t: Float[Tensor, "batch 1"],
) -> Float[Tensor, "batch 1"]:
    return torch.ones_like(t)


def estimate_weighted_denoising_loss_curve(
    *,
    model: BimodalRoundtripModel,
    clean_latents: Float[Tensor, "batch 2"],
    noise_levels: tuple[float, ...],
    decoder_attenuation: float,
    num_noise_draws: int,
) -> list[float]:
    denoising_losses: list[float] = []
    for noise_level in noise_levels:
        t = torch.full(
            (clean_latents.shape[0], 1),
            fill_value=float(noise_level),
            device=clean_latents.device,
            dtype=clean_latents.dtype,
        )
        alpha = latent_noise_weight(t=t)
        accumulated_loss = clean_latents.new_tensor(0.0)
        for _ in range(num_noise_draws):
            noisy_latents = (1.0 - t) * clean_latents + t * torch.randn_like(
                clean_latents
            )
            denoised_latents = model.roundtrip_with_decoder_attenuation(
                y=noisy_latents,
                t=t,
                attenuation=decoder_attenuation,
            )
            accumulated_loss = accumulated_loss + (
                alpha * (denoised_latents - clean_latents).square()
            ).mean()
        denoising_losses.append(float((accumulated_loss / num_noise_draws).item()))
    return denoising_losses


def relabel_denoising_figure(*, figure, step: int) -> object:
    figure.update_layout(
        title=(
            f"estimated denoising loss by noise level at step {step} "
            f"({NOISE_WEIGHTING_LABEL} weighting)"
        )
    )
    figure.update_yaxes(title_text="E[w(t) ||f(y_t, t) - y_0||^2], w(t)=1")
    return figure


def compute_latent_snapshot_drift(
    *,
    model: BimodalRoundtripModel,
    latent_points: Float[Tensor, "batch 2"],
) -> Float[Tensor, "batch 2"]:
    snapshot_t = torch.full(
        (latent_points.shape[0], 1),
        0.35,
        device=latent_points.device,
        dtype=latent_points.dtype,
    )
    latent_base = latent_points.detach().requires_grad_(True)
    latent_t = (1.0 - snapshot_t) * latent_base
    alpha = latent_noise_weight(t=snapshot_t)
    score_mismatch = alpha * (
        model.roundtrip(y=latent_t, t=snapshot_t, frozen=True)
        - analytic_gaussian_denoiser(y_t=latent_t, t=snapshot_t)
    ).square().mean(dim=-1, keepdim=True)
    return -torch.autograd.grad(score_mismatch.sum(), latent_base)[0]


def compute_losses(
    *,
    model: BimodalRoundtripModel,
    x: Float[Tensor, "batch 2"],
    config: ExperimentConfig,
) -> dict[str, Tensor]:
    batch_size = int(x.shape[0])
    y_data = model.encode_zero(x=x)
    y_target = y_data.detach()
    x_recon = model.decode_with_decoder_attenuation(
        y=y_data,
        attenuation=config.train.decoder_attenuation,
    )
    x_cycle_data = model.decode_with_decoder_attenuation(
        y=y_target,
        attenuation=config.train.decoder_attenuation,
    )
    y_cycle_data = model.encode_zero(x=x_cycle_data)
    x_cycle_data_score = model.decode(y=y_target)

    z = torch.randn(batch_size, 2, device=x.device, dtype=x.dtype)
    x_cycle_prior = model.decode_with_decoder_attenuation(
        y=z,
        attenuation=config.train.decoder_attenuation,
    )
    z_cycle = model.encode_zero(x=x_cycle_prior)

    t = sample_time(
        batch_size=batch_size,
        loss_config=config.loss,
        device=DEVICE,
        dtype=torch.float32,
    )
    alpha = latent_noise_weight(t=t)

    eps = torch.randn_like(y_target)
    y_t = (1.0 - t) * y_target + t * eps
    y_denoised = model.roundtrip_with_decoder_attenuation(
        y=y_t,
        t=t,
        attenuation=config.train.decoder_attenuation,
    )

    y_roundtrip = model.encode_zero(x=x_cycle_data_score, frozen=True)
    y_latent = y_roundtrip.detach().requires_grad_(True)
    y_latent_t = (1.0 - t) * y_latent + t * torch.randn_like(y_latent)
    score_mismatch = alpha * (
        model.roundtrip(y=y_latent_t, t=t, frozen=True)
        - analytic_gaussian_denoiser(y_t=y_latent_t, t=t)
    ).square().mean(dim=-1, keepdim=True)
    score_drift = -torch.autograd.grad(score_mismatch.sum(), y_latent)[0]
    score_target = (y_roundtrip.detach() + score_drift.detach()).detach()

    prior_loss = prior_matching_loss(y_data=y_data)
    reconstruction_loss = F.mse_loss(x_recon, x)
    cycle_data_loss = F.mse_loss(y_cycle_data, y_target)
    cycle_prior_loss = F.mse_loss(z_cycle, z)
    denoising_loss = (alpha * (y_denoised - y_target).square()).mean()
    score_loss = F.mse_loss(y_roundtrip, score_target)

    total_loss = (
        config.loss.reconstruction_weight * reconstruction_loss
        + config.loss.prior_matching_weight * prior_loss
        + config.loss.cycle_data_weight * cycle_data_loss
        + config.loss.cycle_prior_weight * cycle_prior_loss
        + config.loss.denoising_weight * denoising_loss
        + config.loss.score_weight * score_loss
    )
    return {
        "total_loss": total_loss,
        "reconstruction_loss": reconstruction_loss,
        "prior_matching_loss": prior_loss,
        "cycle_data_loss": cycle_data_loss,
        "cycle_prior_loss": cycle_prior_loss,
        "denoising_loss": denoising_loss,
        "score_loss": score_loss,
    }


def evaluate_model(
    *,
    model: BimodalRoundtripModel,
    fixed_eval_x: Float[Tensor, "batch 2"],
    fixed_eval_mode_ids: Tensor,
    fixed_prior_points: Float[Tensor, "batch 2"],
    fixed_contour_ids: Tensor,
    fixed_latent_x: Float[Tensor, "batch 2"],
    fixed_latent_mode_ids: Tensor,
    mode_centers: Float[Tensor, "2 2"],
    config: ExperimentConfig,
    step: int,
) -> tuple[dict[str, object], dict[str, object]]:
    model_was_training = model.training
    model.eval()
    with torch.no_grad():
        with torch.cuda.device(DEVICE), torch.device(DEVICE):
            reconstruction_points = model.decode(y=model.encode_zero(x=fixed_eval_x))
            model_points = model.decode(y=fixed_prior_points)
            random_model_points = model.decode(
                y=torch.randn(
                    config.plot.eval_size,
                    2,
                    device=DEVICE,
                    dtype=torch.float32,
                )
            )
            latent_points = model.encode_zero(x=fixed_latent_x)
            pullback_reconstruction_points = model.decode(y=latent_points)
            score_snapshots = build_latent_score_snapshots(
                model=model,
                latent_points=latent_points,
                latent_mode_ids=fixed_latent_mode_ids,
                noise_levels=SCORE_NOISE_LEVELS,
                arrow_stride=config.plot.arrow_stride,
                arrow_display_length=SCORE_ARROW_DISPLAY_LENGTH,
            )
            denoising_loss_estimates = estimate_weighted_denoising_loss_curve(
                model=model,
                clean_latents=latent_points,
                noise_levels=DENOISING_EVAL_LEVELS,
                decoder_attenuation=config.train.decoder_attenuation,
                num_noise_draws=DENOISING_EVAL_NOISE_DRAWS,
            )
    with torch.cuda.device(DEVICE), torch.device(DEVICE):
        latent_drift = compute_latent_snapshot_drift(
            model=model,
            latent_points=latent_points,
        )
        pullback_target_points = model.decode(y=latent_points + latent_drift)

    latent_arrow_vectors, latent_arrow_magnitudes = normalize_vectors_for_display(
        vectors=latent_drift,
        display_length=LATENT_DRIFT_ARROW_DISPLAY_LENGTH,
    )
    pullback_vectors = pullback_target_points - pullback_reconstruction_points
    pullback_arrow_vectors, pullback_arrow_magnitudes = normalize_vectors_for_display(
        vectors=pullback_vectors,
        display_length=PULLBACK_ARROW_DISPLAY_LENGTH,
    )

    arrow_indices = torch.arange(
        0,
        latent_points.shape[0],
        config.plot.arrow_stride,
        device=latent_points.device,
    )
    if model_was_training:
        model.train()

    metrics = {
        "reconstruction_mse": F.mse_loss(reconstruction_points, fixed_eval_x).item(),
        "random_model_mean": random_model_points.mean(dim=0).detach().cpu(),
        "random_model_std": random_model_points.std(dim=0).detach().cpu(),
    }
    snapshot = {
        "step": step,
        "data_points": fixed_eval_x.detach().cpu(),
        "reconstruction_points": reconstruction_points.detach().cpu(),
        "data_mode_ids": fixed_eval_mode_ids.detach().cpu(),
        "model_points": model_points.detach().cpu(),
        "contour_ids": fixed_contour_ids.detach().cpu(),
        "contour_levels": config.plot.contour_levels,
        "mode_centers": mode_centers.detach().cpu(),
        "latent_points": latent_points.detach().cpu(),
        "latent_mode_ids": fixed_latent_mode_ids.detach().cpu(),
        "latent_arrow_points": latent_points[arrow_indices].detach().cpu(),
        "latent_arrow_vectors": latent_arrow_vectors[arrow_indices].detach().cpu(),
        "latent_arrow_mode_ids": fixed_latent_mode_ids[arrow_indices].detach().cpu(),
        "latent_arrow_magnitudes": latent_arrow_magnitudes[arrow_indices]
        .detach()
        .cpu(),
        "score_snapshots": score_snapshots,
        "pullback_reconstruction_points": pullback_reconstruction_points.detach().cpu(),
        "pullback_arrow_points": pullback_reconstruction_points[arrow_indices]
        .detach()
        .cpu(),
        "pullback_arrow_vectors": pullback_arrow_vectors[arrow_indices].detach().cpu(),
        "pullback_arrow_mode_ids": fixed_latent_mode_ids[arrow_indices].detach().cpu(),
        "pullback_arrow_magnitudes": pullback_arrow_magnitudes[arrow_indices]
        .detach()
        .cpu(),
        "denoising_eval_levels": DENOISING_EVAL_LEVELS,
        "denoising_loss_estimates": denoising_loss_estimates,
    }
    return snapshot, metrics


def initialize_history() -> dict[str, list[float]]:
    return {
        "step": [],
        "total_loss": [],
        "reconstruction_loss": [],
        "prior_matching_loss": [],
        "cycle_data_loss": [],
        "cycle_prior_loss": [],
        "denoising_loss": [],
        "score_loss": [],
        "reconstruction_mse": [],
    }


def build_snapshot_figures(
    *,
    history: dict[str, list[float]],
    snapshot: dict[str, object],
) -> dict[str, object]:
    step = int(snapshot["step"])
    return {
        "losses": plot_training_history(
            history=history,
            step=step,
        ),
        "snapshot": plot_bimodal_snapshot(
            data_points=snapshot["data_points"],
            reconstruction_points=snapshot["reconstruction_points"],
            data_mode_ids=snapshot["data_mode_ids"],
            model_points=snapshot["model_points"],
            contour_ids=snapshot["contour_ids"],
            contour_levels=snapshot["contour_levels"],
            mode_centers=snapshot["mode_centers"],
            latent_points=snapshot["latent_points"],
            latent_mode_ids=snapshot["latent_mode_ids"],
            latent_arrow_points=snapshot["latent_arrow_points"],
            latent_arrow_vectors=snapshot["latent_arrow_vectors"],
            latent_arrow_mode_ids=snapshot["latent_arrow_mode_ids"],
            latent_arrow_magnitudes=snapshot["latent_arrow_magnitudes"],
            step=step,
        ),
        "score_snapshot": plot_latent_score_snapshot_selector(
            score_snapshots=snapshot["score_snapshots"],
            step=step,
        ),
        "pullback": plot_pullback_snapshot(
            reconstruction_points=snapshot["pullback_reconstruction_points"],
            pullback_arrow_points=snapshot["pullback_arrow_points"],
            pullback_arrow_vectors=snapshot["pullback_arrow_vectors"],
            pullback_arrow_mode_ids=snapshot["pullback_arrow_mode_ids"],
            pullback_arrow_magnitudes=snapshot["pullback_arrow_magnitudes"],
            mode_centers=snapshot["mode_centers"],
            step=step,
        ),
        "denoising_sweep": relabel_denoising_figure(
            figure=plot_denoising_loss_curve(
                noise_levels=snapshot["denoising_eval_levels"],
                denoising_losses=snapshot["denoising_loss_estimates"],
                step=step,
            ),
            step=step,
        ),
    }


def save_snapshot_artifacts(
    *,
    figures: dict[str, object],
    step: int,
) -> Path:
    step_dir = RUN_ROOT / f"{step:07d}"
    for artifact_name, figure in figures.items():
        save_plotly_figure(
            figure=figure,
            path=step_dir / f"{artifact_name}.html",
        )
    return step_dir


def display_snapshot_artifacts(
    *,
    figures: dict[str, object],
    step: int,
    step_dir: Path,
) -> None:
    print(f"step {step}")
    print(f"artifacts: {step_dir}")
    display(figures["losses"])
    display(figures["snapshot"])
    display(figures["score_snapshot"])
    display(figures["pullback"])
    display(figures["denoising_sweep"])


def train_model(
    *,
    config: ExperimentConfig,
    fixed_eval_x: Float[Tensor, "batch 2"],
    fixed_eval_mode_ids: Tensor,
    fixed_prior_points: Float[Tensor, "batch 2"],
    fixed_contour_ids: Tensor,
    fixed_latent_x: Float[Tensor, "batch 2"],
    fixed_latent_mode_ids: Tensor,
    mode_centers: Float[Tensor, "2 2"],
) -> dict[str, object]:
    model = BimodalRoundtripModel().to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.train.lr,
        weight_decay=config.train.weight_decay,
    )
    history = initialize_history()
    final_snapshot: dict[str, object] = {}
    final_metrics: dict[str, object] = {}

    progress_bar = tqdm(
        range(1, config.train.steps + 1),
        desc="training",
        unit="step",
    )
    for step in progress_bar:
        x, _ = sample_bimodal_gaussian(
            batch_size=config.data.batch_size,
            data_config=config.data,
            device=DEVICE,
            dtype=torch.float32,
        )
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.device(DEVICE), torch.device(DEVICE):
            loss_terms = compute_losses(model=model, x=x, config=config)
        loss_terms["total_loss"].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.train.grad_clip)
        optimizer.step()

        should_snapshot = (
            step % config.plot.snapshot_every_steps == 0 or step == config.train.steps
        )
        should_log = (
            step == 1
            or step % config.train.log_every_steps == 0
            or step == config.train.steps
            or should_snapshot
        )
        if should_log:
            snapshot, metrics = evaluate_model(
                model=model,
                fixed_eval_x=fixed_eval_x,
                fixed_eval_mode_ids=fixed_eval_mode_ids,
                fixed_prior_points=fixed_prior_points,
                fixed_contour_ids=fixed_contour_ids,
                fixed_latent_x=fixed_latent_x,
                fixed_latent_mode_ids=fixed_latent_mode_ids,
                mode_centers=mode_centers,
                config=config,
                step=step,
            )
            final_snapshot = snapshot
            final_metrics = metrics
            history["step"].append(step)
            history["total_loss"].append(float(loss_terms["total_loss"].item()))
            history["reconstruction_loss"].append(
                float(loss_terms["reconstruction_loss"].item())
            )
            history["prior_matching_loss"].append(
                float(loss_terms["prior_matching_loss"].item())
            )
            history["cycle_data_loss"].append(
                float(loss_terms["cycle_data_loss"].item())
            )
            history["cycle_prior_loss"].append(
                float(loss_terms["cycle_prior_loss"].item())
            )
            history["denoising_loss"].append(float(loss_terms["denoising_loss"].item()))
            history["score_loss"].append(float(loss_terms["score_loss"].item()))
            history["reconstruction_mse"].append(
                float(final_metrics["reconstruction_mse"])
            )
            progress_bar.set_postfix(
                total=f"{history['total_loss'][-1]:.3f}",
                recon=f"{history['reconstruction_loss'][-1]:.3f}",
                prior=f"{history['prior_matching_loss'][-1]:.3f}",
                denoise=f"{history['denoising_loss'][-1]:.3f}",
                score=f"{history['score_loss'][-1]:.3f}",
                recon_mse=f"{history['reconstruction_mse'][-1]:.4f}",
            )
        if should_snapshot:
            figures = build_snapshot_figures(
                history=history,
                snapshot=final_snapshot,
            )
            step_dir = save_snapshot_artifacts(
                figures=figures,
                step=int(final_snapshot["step"]),
            )
            display_snapshot_artifacts(
                figures=figures,
                step=int(final_snapshot["step"]),
                step_dir=step_dir,
            )

    return {
        "config": config,
        "model": model,
        "history": history,
        "snapshot": final_snapshot,
        "metrics": final_metrics,
        "run_catalog": RUN_CATALOG,
        "run_root": RUN_ROOT,
    }


In [3]:
FIXED_EVAL_X, FIXED_EVAL_MODE_IDS = sample_bimodal_gaussian(
    batch_size=CONFIG.plot.eval_size,
    data_config=CONFIG.data,
    device=DEVICE,
    dtype=torch.float32,
)
FIXED_PRIOR_POINTS, FIXED_CONTOUR_IDS = build_gaussian_contour_points(
    contour_levels=CONFIG.plot.contour_levels,
    points_per_contour=CONFIG.plot.contour_points_per_level,
    device=DEVICE,
    dtype=torch.float32,
)
FIXED_LATENT_X, FIXED_LATENT_MODE_IDS = sample_bimodal_gaussian(
    batch_size=CONFIG.plot.latent_plot_size,
    data_config=CONFIG.data,
    device=DEVICE,
    dtype=torch.float32,
)
MODE_CENTERS = mode_centers_tensor(
    data_config=CONFIG.data,
    device=DEVICE,
    dtype=torch.float32,
)

preview_model = BimodalRoundtripModel().to(DEVICE)
preview_snapshot, preview_metrics = evaluate_model(
    model=preview_model,
    fixed_eval_x=FIXED_EVAL_X,
    fixed_eval_mode_ids=FIXED_EVAL_MODE_IDS,
    fixed_prior_points=FIXED_PRIOR_POINTS,
    fixed_contour_ids=FIXED_CONTOUR_IDS,
    fixed_latent_x=FIXED_LATENT_X,
    fixed_latent_mode_ids=FIXED_LATENT_MODE_IDS,
    mode_centers=MODE_CENTERS,
    config=CONFIG,
    step=0,
)
preview_figures = build_snapshot_figures(
    history=initialize_history(),
    snapshot=preview_snapshot,
)
preview_step_dir = save_snapshot_artifacts(
    figures=preview_figures,
    step=int(preview_snapshot["step"]),
)
print(preview_metrics)
print(f"preview artifacts: {preview_step_dir}")
display(preview_figures["snapshot"])
display(preview_figures["score_snapshot"])
display(preview_figures["pullback"])
display(preview_figures["denoising_sweep"])


{'reconstruction_mse': 12.751707077026367, 'random_model_mean': tensor([ 0.0268, -0.0137]), 'random_model_std': tensor([0.0378, 0.0226])}
preview artifacts: /tmp/20260321-125751-0de54aed5d04/0000000


In [4]:
results = train_model(
    config=CONFIG,
    fixed_eval_x=FIXED_EVAL_X,
    fixed_eval_mode_ids=FIXED_EVAL_MODE_IDS,
    fixed_prior_points=FIXED_PRIOR_POINTS,
    fixed_contour_ids=FIXED_CONTOUR_IDS,
    fixed_latent_x=FIXED_LATENT_X,
    fixed_latent_mode_ids=FIXED_LATENT_MODE_IDS,
    mode_centers=MODE_CENTERS,
)
results["metrics"]

training:   0%|          | 0/1280000 [00:00<?, ?step/s]

step 4000
artifacts: /tmp/20260321-125751-0de54aed5d04/0004000


step 8000
artifacts: /tmp/20260321-125751-0de54aed5d04/0008000


step 12000
artifacts: /tmp/20260321-125751-0de54aed5d04/0012000


step 16000
artifacts: /tmp/20260321-125751-0de54aed5d04/0016000


KeyboardInterrupt: 

In [ ]:
final_figures = build_snapshot_figures(
    history=results["history"],
    snapshot=results["snapshot"],
)
final_step_dir = save_snapshot_artifacts(
    figures=final_figures,
    step=int(results["snapshot"]["step"]),
)
print(f"final artifacts: {final_step_dir}")
display(final_figures["losses"])
display(final_figures["snapshot"])
display(final_figures["score_snapshot"])
display(final_figures["pullback"])
display(final_figures["denoising_sweep"])
